# Omni Claw 一 测试


In [ ]:
from __future__ import annotations

import sqlite3
import subprocess
import sys
from pathlib import Path
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

ROOT = Path.cwd()
if ROOT.name == 'test':
    ROOT = ROOT.parent

print('项目根目录:', ROOT)

def run_cmd(args: list[str], cwd: Path = ROOT) -> subprocess.CompletedProcess:
    result = subprocess.run(
        args,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        encoding="utf-8",   # 关键
        errors="replace",   # 避免因个别字符中断
    )
    print("> 命令:", " ".join(args))
    print("> 返回码:", result.returncode)
    if result.stdout:
        print("> stdout:\n", result.stdout)
    if result.stderr:
        print("> stderr:\n", result.stderr)
    return result


项目根目录: d:\code\program\OmniClaw


## 1.1 工程初始化测试（CLI）

In [4]:
cli_commands = [
    [sys.executable, '-m', 'backend.app.cli', '--help'],
    [sys.executable, '-m', 'backend.app.cli', 'run'],
    [sys.executable, '-m', 'backend.app.cli', 'config'],
    [sys.executable, '-m', 'backend.app.cli', 'monitor'],
    [sys.executable, '-m', 'backend.app.cli', 'tasks'],
]

for cmd in cli_commands:
    r = run_cmd(cmd)
    assert r.returncode == 0, f'CLI 命令失败: {cmd}'

print('1.1 CLI 测试通过')


> 命令: d:\code\program\OmniClaw\.venv\Scripts\python.exe -m backend.app.cli --help
> 返回码: 0
> stdout:
                                                                                
 Usage: python -m backend.app.cli [OPTIONS] COMMAND [ARGS]...                  
                                                                               
 Omni Claw CLI                                                                 
                                                                               
┌─ Options ───────────────────────────────────────────────────────────────────┐
│ --install-completion          Install completion for the current shell.     │
│ --show-completion             Show completion for the current shell, to     │
│                               copy it or customize the installation.        │
│ --help                        Show this message and exit.                   │
└─────────────────────────────────────────────────────────────────────────────┘
┌─ Commands ──────

## 1.2 API 路由测试（不启动外部服务，直接用 TestClient）

In [5]:
from fastapi.testclient import TestClient
from backend.app.main import app

with TestClient(app) as client:
    health_resp = client.get('/health')
    runtime_resp = client.get('/runtime/status')

assert health_resp.status_code == 200
assert runtime_resp.status_code == 200
assert health_resp.json().get('status') == 'ok'
assert 'status' in runtime_resp.json()

print('health:', health_resp.json())
print('runtime/status:', runtime_resp.json())
print('1.2 API 测试通过')


health: {'status': 'ok'}
runtime/status: {'status': 'idle', 'message': 'runtime router is ready'}
1.2 API 测试通过


## 1.3 配置系统测试

In [6]:
from backend.app.core.config import get_settings

s = get_settings()
print('provider:', s.provider)
print('model_name:', s.model_name)
print('openai_api_base:', getattr(s, 'openai_api_base', None))

assert isinstance(s.provider, str) and len(s.provider) > 0
assert isinstance(s.model_name, str) and len(s.model_name) > 0
print('1.3 配置测试通过')


provider: aliyun
model_name: qwen-max
openai_api_base: https://dashscope.aliyuncs.com/compatible-mode/v1
1.3 配置测试通过


## 1.4 SQLite 初始化测试

In [8]:
from backend.app.storage.sqlite import init_db

db_path = ROOT / "workspace" / "omniclaw.db"
await init_db(str(db_path))
assert db_path.exists(), "数据库文件未创建"

conn = sqlite3.connect(db_path)
cur = conn.cursor()
tables = [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
conn.close()

print('查表结果:', tables)
for need in ['sessions', 'messages', 'tasks', 'audit_events']:
    assert need in tables, f'缺少数据表: {need}'

print('1.4 SQLite 测试通过')


查表结果: ['audit_events', 'messages', 'sessions', 'sqlite_sequence', 'tasks']
1.4 SQLite 测试通过


## 1.5 数据模型（Schema）测试

In [9]:
from backend.app.models.schemas import TaskCreate, MessageCreate

task = TaskCreate(subject='测试任务')
msg = MessageCreate(session_id='s-1', role='user', content='hello')

print('TaskCreate:', task.model_dump())
print('MessageCreate:', msg.model_dump())

assert task.subject == '测试任务'
assert msg.role == 'user'
print('1.5 Schema 测试通过')


TaskCreate: {'subject': '测试任务', 'description': '', 'owner': None, 'blocked_by': [], 'target_time': None}
MessageCreate: {'session_id': 's-1', 'role': 'user', 'content': 'hello', 'tool_name': None}
1.5 Schema 测试通过


## 1.6 JSONL 日志测试

In [10]:
from backend.app.core.logger import LOG_FILE, log_event, read_recent_events

before = 0
if LOG_FILE.exists():
    before = len([x for x in LOG_FILE.read_text(encoding='utf-8').splitlines() if x.strip()])

evt = log_event(
    event='system_action',
    thread_id='test-1-notebook',
    payload={'step': '1.6'},
)

after = len([x for x in LOG_FILE.read_text(encoding='utf-8').splitlines() if x.strip()])
recent = read_recent_events(1)

print('日志文件:', LOG_FILE)
print('before/after:', before, after)
print('recent:', recent)

assert after == before + 1, '日志行数未增加'
assert recent and recent[-1]['id'] == evt['id'], '最近日志读取不匹配'
print('1.6 日志测试通过')


日志文件: .logs\audit.jsonl
before/after: 0 1
recent: [{'id': '445db65d-b2ad-4824-8c36-fc1ba201378e', 'ts': '2026-04-23T12:09:02.339793+00:00', 'event': 'system_action', 'thread_id': 'test-1-notebook', 'trace_id': '4a59d501-9554-4230-abbf-e5cf36d21f6c', 'payload': {'step': '1.6'}}]
1.6 日志测试通过


## 1.7 项目回归脚本测试

In [11]:
check_script = ROOT / 'scripts' / '1_check.py'
assert check_script.exists(), '缺少 scripts/1_check.py'

r = run_cmd([sys.executable, str(check_script)])
assert r.returncode == 0, '1_check.py 回归失败'
print('1.7 回归脚本测试通过')


> 命令: d:\code\program\OmniClaw\.venv\Scripts\python.exe d:\code\program\OmniClaw\scripts\1_check.py
> 返回码: 0
> stdout:
 === 项目回归检查开始 ===
[PASS] 核心文件存在
[PASS] 数据库初始化幂等
  - 查表结果: ['audit_events', 'messages', 'sessions', 'sqlite_sequence', 'tasks']
[PASS] 数据库与表结构
[PASS] 任务插入与查询
[PASS] 日志写入与读取

=== 检查完成：5/5 通过 ===
项目回归通过：存在性 + 行为性检查全部通过。

1.7 回归脚本测试通过
